<h1><strong>Problem Statement</strong></h1>


<h2><strong>Business Context</strong></h2>


<p>Workplace safety in hazardous environments like construction sites and industrial plants is crucial to prevent accidents and injuries. One of the most important safety measures is ensuring workers wear safety helmets, which protect against head injuries from falling objects and machinery. Non-compliance with helmet regulations increases the risk of serious injuries or fatalities, making effective monitoring essential, especially in large-scale operations where manual oversight is prone to errors and inefficiency.</p>
<p>To overcome these challenges, SafeGuard Corp plans to develop an automated image analysis system capable of detecting whether workers are wearing safety helmets. This system will improve safety enforcement, ensuring compliance and reducing the risk of head injuries. By automating helmet monitoring, SafeGuard aims to enhance efficiency, scalability, and accuracy, ultimately fostering a safer work environment while minimizing human error in safety oversight.</p>


<h2><strong>Objective</strong></h2>


<p>As a data scientist at SafeGuard Corp, you are tasked with developing an image classification model that classifies images into one of two categories:</p>
<ul>
<li><strong>With Helmet:</strong> Workers wearing safety helmets.</li>
<li><strong>Without Helmet:</strong> Workers not wearing safety helmets.</li>
</ul>


<h2><strong>Data Description</strong></h2>


<p>The dataset consists of <strong>631 images</strong>, equally divided into two categories:</p>
<ul>
<li><strong>With Helmet:</strong> 311 images showing workers wearing helmets.</li>
<li><strong>Without Helmet:</strong> 320 images showing workers not wearing helmets.</li>
</ul>
<p><strong>Dataset Characteristics:</strong></p>
<ul>
<li><strong>Variations in Conditions:</strong> Images include diverse environments such as construction sites, factories, and industrial settings, with variations in lighting, angles, and worker postures to simulate real-world conditions.</li>
<li><strong>Worker Activities:</strong> Workers are depicted in different actions such as standing, using tools, or moving, ensuring robust model learning for various scenarios.</li>
</ul>


<h1><strong>Installing and Importing the Necessary Libraries</strong></h1>


In [ ]:
!pip install tensorflow[and-cuda] scikit-learn==1.6.1 opencv-python==4.12.0.88 seaborn==0.13.2 matplotlib==3.10.0 numpy==2.0.2 pandas==2.2.2 -q


In [ ]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.__version__)


<p><strong>Note:</strong></p>
<ul>
<li><p>After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.</p>
</li>
<li><p>On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.</p>
</li>
</ul>


In [ ]:
import os
import random
import numpy as np                                                                               # Importing numpy for Matrix Operations
import pandas as pd
import seaborn as sns
import matplotlib.image as mpimg                                                                              # Importing pandas to read CSV files
import matplotlib.pyplot as plt                                                                  # Importting matplotlib for Plotting and visualizing images
import math                                                                                      # Importing math module to perform mathematical operations
import cv2

# Tensorflow modules
import keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator                              # Importing the ImageDataGenerator for data augmentation
from tensorflow.keras.models import Sequential                                                   # Importing the sequential module to define a sequential model
from tensorflow.keras.layers import Dense,Dropout,Flatten,Conv2D,MaxPooling2D,BatchNormalization # Defining all the layers to build our CNN Model
from tensorflow.keras.optimizers import Adam,SGD                                                 # Importing the optimizers which can be used in our model
from sklearn import preprocessing                                                                # Importing the preprocessing module to preprocess the data
from sklearn.model_selection import train_test_split                                             # Importing train_test_split function to split the data into train and test
from sklearn.metrics import confusion_matrix
from tensorflow.keras.models import Model
from keras.applications.vgg16 import VGG16                                               # Importing confusion_matrix to plot the confusion matrix

# Display images using OpenCV
from google.colab.patches import cv2_imshow

#Imports functions for evaluating the performance of machine learning models
from sklearn.metrics import confusion_matrix, f1_score,accuracy_score, recall_score, precision_score, classification_report
from sklearn.metrics import mean_squared_error as mse                                                 # Importing cv2_imshow from google.patches to display images

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Set the seed using keras.utils.set_random_seed. This will set:
# 1) `numpy` seed
# 2) backend random seed
# 3) `python` random seed
tf.keras.utils.set_random_seed(812)


<h1><strong>Data Overview</strong></h1>


<h2>Loading the data</h2>


In [ ]:
images = np.load("images_proj.npy")
labels = pd.read_csv("Labels_proj.csv")


In [ ]:
print(images.shape)
print(labels.shape)


In [ ]:
plt.imshow(images[5]);


<h1><strong>Exploratory Data Analysis</strong></h1>


In [ ]:
# Converting the images from BGR to RGB using cvtColor function of OpenCV
for i in range(len(images)):
  images[i] = cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB)


<h3>Plot random images from each of the classes and print their corresponding labels.</h3>


In [ ]:
# Show sample images
# Convert to NumPy array if not already
X = np.array(images)
y = np.array(labels)

# Find indices for each class
helmet_indices = np.where(y == 1)[0]
no_helmet_indices = np.where(y == 0)[0]

# Pick 5 random indices from each
helmet_samples = random.sample(list(helmet_indices), 5)
no_helmet_samples = random.sample(list(no_helmet_indices), 5)

# Plot images without helmet
plt.figure(figsize=(15, 4))
for i, idx in enumerate(no_helmet_samples):
    plt.subplot(2, 5, i+1)
    plt.imshow(X[idx].squeeze(), cmap='gray' if X.shape[-1] == 1 else None) # Ensures any grayscale images are shown correctly
    plt.title("No Helmet")
    plt.axis('off')

# Plot images with helmet
for i, idx in enumerate(helmet_samples):
    plt.subplot(2, 5, i+6)
    plt.imshow(X[idx].squeeze(), cmap='gray' if X.shape[-1] == 1 else None)
    plt.title("With Helmet")
    plt.axis('off')

plt.suptitle("Sample Images by Class", fontsize=16)
plt.tight_layout()
plt.show()


<h2>Checking for class imbalance</h2>


In [ ]:
# Plot the class distribution with Seaborn countplot.
sns.countplot(x=labels['Label'])
plt.title("Class Distribution: With Helmet (1) vs Without Helmet (0)")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()


<p>Observations:</p>
<ol>
<li>I had to convert images from BGR to RBG to</li>
<li>List item</li>
</ol>


In [ ]:
# 1. Check Class Distribution
class_counts = labels['Label'].value_counts()
class_percentages = labels['Label'].value_counts(normalize=True) * 100

print("--- Class Distribution ---")
print(class_counts)
print(f"\nWith Helmet (1): {class_percentages[1]:.2f}%")
print(f"Without Helmet (0): {class_percentages[0]:.2f}%")

# 2. Analyze Image Dimensions
heights = [img.shape[0] for img in images]
widths = [img.shape[1] for img in images]
channels = [img.shape[2] for img in images]

print("\n--- Image Dimensions ---")
print(f"Unique Heights: {set(heights)}")
print(f"Unique Widths: {set(widths)}")
print(f"Unique Channels: {set(channels)}")


<h3><strong>Key EDA Findings:</strong></h3><ol>
<li><strong>Balanced Dataset:</strong> The dataset is extremely well-balanced with 311 images of workers <em>With Helmets</em> and 320 images <em>Without Helmets</em>. This means we don't need to worry about class weight adjustments or oversampling.</li>
<li><strong>Uniform Dimensions:</strong> All images are pre-resized to a consistent 200x200 RGB format. This simplifies the pipeline as no complex padding or cropping is required initially.</li>
<li><strong>Color Space:</strong> The images were originally in BGR format (common with OpenCV) and were correctly converted to RGB for standard deep learning libraries.</li>
</ol>


<h1><strong>Data Preprocessing</strong></h1>


<h2>Converting images to grayscale</h2>


<p>I am not going to convert to grayscale because I feel like we will lose important color information to identift helmets so going to use RGB</p>


<h3>Splitting the dataset</h3>


In [ ]:
# Splitting the dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train set: {X_train.shape[0]}")
print(f"Validation set: {X_val.shape[0]}")
print(f"Test set: {X_test.shape[0]}")


In [ ]:
# 1. Resize images to 224x224 using tf.image.resize (fast and efficient)
X_train_vgg_raw = tf.image.resize(X_train, (224, 224)).numpy()
X_val_vgg_raw   = tf.image.resize(X_val, (224, 224)).numpy()
X_test_vgg_raw  = tf.image.resize(X_test, (224, 224)).numpy()


In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input

# 2. Apply VGG16 specific preprocessing (expects 0-255 pixel values)
X_train_vgg = preprocess_input(X_train_vgg_raw)
X_val_vgg   = preprocess_input(X_val_vgg_raw)
X_test_vgg  = preprocess_input(X_test_vgg_raw)


<h3>Labelling</h3>


In [ ]:
# Convert labels from names to one hot vectors.
# We have already used encoding methods like onehotencoder and labelencoder earlier so now we will be using a new encoding method called labelBinarizer.
# Labelbinarizer works similar to onehotencoder

from sklearn.preprocessing import LabelBinarizer
enc = LabelBinarizer()
y_train_encoded = enc.fit_transform(y_train)
y_val_encoded=enc.transform(y_val)
y_test_encoded=enc.transform(y_test)


In [ ]:
y_train_encoded[0]


<p>I am checking the size of the images to see if i need to resize for using a sample CNN</p>


In [ ]:
# If images are numpy arrays
import numpy as np

# Convert all images to arrays if not already
image_arrays = [np.array(img) for img in images]

# Print min, max, mean dimensions
heights = [img.shape[0] for img in image_arrays]
widths  = [img.shape[1] for img in image_arrays]

print("Image heights: min =", min(heights), "max =", max(heights), "mean =", np.mean(heights))
print("Image widths : min =", min(widths), "max =", max(widths), "mean =", np.mean(widths))


<p>Images sizes seems relatively reasonable to not going to resize for CNN</p>


<h3>Data Normalization</h3>


In [ ]:
# Now we can safely normalize our original arrays for the Simple CNN
X_train = X_train / 255.0
X_val = X_val / 255.0
X_test = X_test / 255.0


In [ ]:
# Visualizing the preprocessing steps already performed in the notebook
import matplotlib.pyplot as plt

# Let's pick a sample index to compare
sample_idx = 0

plt.figure(figsize=(12, 5))

# 1. 'Before' - Image after BGR-to-RGB and 0-1 normalization (Done in cells M-7Jmt25-WGC and GOyE9HKcddXq)
plt.subplot(1, 2, 1)
plt.imshow(X_train[sample_idx])
plt.title("1. Initial Processing\n(RGB + 0-1 Normalization)")
plt.axis('off')

# 2. 'After' - Image after VGG16-specific preprocessing (Done in cell QkHv-9K3iA5F)
# Note: VGG preprocessing subtracts the mean, so we scale it back to 0-1 just for visualization.
vgg_img = X_train_vgg[sample_idx]
vgg_viz = (vgg_img - vgg_img.min()) / (vgg_img.max() - vgg_img.min())

plt.subplot(1, 2, 2)
plt.imshow(vgg_viz)
plt.title("2. VGG16 Final Preprocessing\n(Mean Subtraction Applied)")
plt.axis('off')

plt.tight_layout()
plt.show()


<h1><strong>Model Building</strong></h1>


<h2>Model Evaluation Criterion</h2>


<h2>Utility Functions</h2>


In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using statsmodels
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # checking which probabilities are greater than threshold
    pred = model.predict(predictors).reshape(-1)>0.5

    # Use np.asarray to handle both pandas Series and numpy arrays
    target = np.asarray(target).reshape(-1)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred, average='weighted')  # to compute Recall
    precision = precision_score(target, pred, average='weighted')  # to compute Precision
    f1 = f1_score(target, pred, average='weighted')  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame({"Accuracy": acc, "Recall": recall, "Precision": precision, "F1 Score": f1,},index=[0],)

    return df_perf


In [ ]:
def plot_confusion_matrix(model,predictors,target,ml=False):
    """
    Function to plot the confusion matrix

    model: classifier
    predictors: independent variables
    target: dependent variable
    ml: To specify if the model used is an sklearn ML model or not (True means ML model)
    """

    # checking which probabilities are greater than threshold
    pred = model.predict(predictors).reshape(-1)>0.5

    # Use np.asarray to handle both pandas Series and numpy arrays
    target = np.asarray(target).reshape(-1)

    # Plotting the Confusion Matrix using confusion matrix() function which is also predefined tensorflow module
    cm = confusion_matrix(target, pred)
    f, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        linewidths=.4,
        fmt="d",
        square=True,
        ax=ax
    )
    plt.show()


<h2>Model 1: Simple Convolutional Neural Network (CNN)</h2>


In [ ]:
cnn_model_1 = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(200, 200, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])


In [ ]:
cnn_model_1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model_1.summary()


In [ ]:
history_1 = cnn_model_1.fit(
    X_train, y_train,  # Using standard labels, no Binarizer needed
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=2
)


<h3>Vizualizing the predictions</h3>


In [ ]:
# Plot accuracy
plt.plot(history_1.history['accuracy'], label='Train')
plt.plot(history_1.history['val_accuracy'], label='Validation')
plt.title('Model 1 Accuracy')
plt.legend()
plt.show()


In [ ]:
# Model 1 Evaluation
print("Train performance metrics:")
display(model_performance_classification(cnn_model_1, X_train, y_train))
plot_confusion_matrix(cnn_model_1, X_train, y_train)

print("Validation performance metrics:")
display(model_performance_classification(cnn_model_1, X_val, y_val))
plot_confusion_matrix(cnn_model_1, X_val, y_val)


In [ ]:
# Visualize and predict for some random index to cross verify
index = 60
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

# Make prediction: Reshape to (1, 200, 200, 3) to add batch dimension
prediction = cnn_model_1.predict(X_val[index].reshape(1, 200, 200, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])

# Visualize and predict for another random index
index = 0
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

prediction = cnn_model_1.predict(X_val[index].reshape(1, 200, 200, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])

# Visualize and predict for another random index
index = 10
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

prediction = cnn_model_1.predict(X_val[index].reshape(1, 200, 200, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])


<p>We can see that the model predicted 2 out of the 3 correctly.</p>


<h2>Model 2: (VGG-16 (Base))</h2>


In [ ]:
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


In [ ]:
# Freeze the VGG layers
for layer in vgg_base.layers:
    layer.trainable = False


In [ ]:
model_2 = Sequential([
    vgg_base,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])


In [ ]:
model_2.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_2.summary()


In [ ]:
# Note how we are using X_train_vgg and y_train here!
history_2 = model_2.fit(
    X_train_vgg, y_train,
    validation_data=(X_val_vgg, y_val),
    epochs=10,
    batch_size=32,
    verbose=2
)


<h3>Visualizing the prediction:</h3>


In [ ]:
# Plot accuracy
plt.plot(history_2.history['accuracy'], label='Train')
plt.plot(history_2.history['val_accuracy'], label='Validation')
plt.title('Model 2 (VGG-16) Accuracy')
plt.legend()
plt.show()


In [ ]:
# Model 2 Evaluation
print("Train performance metrics (VGG-16):")
display(model_performance_classification(model_2, X_train_vgg, y_train))
plot_confusion_matrix(model_2, X_train_vgg, y_train)

print("Validation performance metrics (VGG-16):")
display(model_performance_classification(model_2, X_val_vgg, y_val))
plot_confusion_matrix(model_2, X_val_vgg, y_val)


In [ ]:
# Visualize and predict for some random index to cross verify
# Note: We use X_val for display (0-1 range) but X_val_vgg for the model prediction
index = 60
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

# Make prediction using the correctly preprocessed VGG array
prediction = model_2.predict(X_val_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])

# Visualize and predict for another random index
index = 0
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

prediction = model_2.predict(X_val_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])

# Visualize and predict for another random index
index = 10
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

prediction = model_2.predict(X_val_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])


<p>We can see that even with VGG the 3rd image is still not corrected.</p>


<h2>Model 3: (VGG-16 (Base + FFNN))</h2>


In [ ]:
vgg_base_3 = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg_base_3.trainable = False


In [ ]:
# 2. Build the Model with a more robust FFNN
model_3 = Sequential([
    vgg_base_3,
    Flatten(),

    # First Dense Block
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),

    # Second Dense Block
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    # Output Layer
    Dense(1, activation='sigmoid')
])


In [ ]:
model_3.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_3.summary()


In [ ]:
# 4. Train
history_3 = model_3.fit(
    X_train_vgg, y_train,
    validation_data=(X_val_vgg, y_val),
    epochs=15, # Increased epochs slightly to let the deeper FFNN learn
    batch_size=32,
    verbose=1
)


<h4>Visualizing the predictions</h4>


In [ ]:
# Plot accuracy
plt.plot(history_3.history['accuracy'], label='Train')
plt.plot(history_3.history['val_accuracy'], label='Validation')
plt.title('Model 3 (VGG-16) Accuracy')
plt.legend()
plt.show()


In [ ]:
print("Model 3: Train Performance")
display(model_performance_classification(model_3, X_train_vgg, y_train))
plot_confusion_matrix(model_3, X_train_vgg, y_train)

print("Model 3: Validation Performance")
display(model_performance_classification(model_3, X_val_vgg, y_val))

plot_confusion_matrix(model_3, X_val_vgg, y_val)


<p>Accuracy has started to become 1.0</p>


<h2>Model 4: (VGG-16 (Base + FFNN + Data Augmentation)</h2>


<ul>
<li><p>In most of the real-world case studies, it is challenging to acquire a large number of images and then train CNNs.</p>
</li>
<li><p>To overcome this problem, one approach we might consider is <strong>Data Augmentation</strong>.</p>
</li>
<li><p>CNNs have the property of <strong>translational invariance</strong>, which means they can recognise an object even if its appearance shifts translationally in some way. - Taking this attribute into account, we can augment the images using the techniques listed below</p>
<ul>
<li>Horizontal Flip (should be set to True/False)</li>
<li>Vertical Flip (should be set to True/False)</li>
<li>Height Shift (should be between 0 and 1)</li>
<li>Width Shift (should be between 0 and 1)</li>
<li>Rotation (should be between 0 and 180)</li>
<li>Shear (should be between 0 and 1)</li>
<li>Zoom (should be between 0 and 1) etc.</li>
</ul>
</li>
</ul>
<p>Remember, <strong>data augmentation should not be used in the validation/test data set</strong>.</p>


In [ ]:
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    preprocessing_function=preprocess_input # Critical: VGG preprocessing applied here
)


In [ ]:
# For validation, we ONLY apply preprocessing (no augmentation)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)


In [ ]:
# We use X_train_vgg_raw (the 0-255 resized images) created in the Preprocessing step
train_generator = train_datagen.flow(X_train_vgg_raw, y_train, batch_size=32)
val_generator = val_datagen.flow(X_val_vgg_raw, y_val, batch_size=32)


In [ ]:
# 3. Build Model 4 (Same robust architecture as Model 3)
vgg_base_4 = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg_base_4.trainable = False


In [ ]:
model_4 = Sequential([
    vgg_base_4,
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])


In [ ]:
model_4.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Train the model using the generator
# Since we have a small dataset, we'll train for more epochs as augmentation makes it harder to overfit
history_4 = model_4.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    verbose=1
)


<h4>Visualizing the predictions</h4>


In [ ]:
# Plot accuracy
plt.plot(history_4.history['accuracy'], label='Train')
plt.plot(history_4.history['val_accuracy'], label='Validation')
plt.title('Model 4 (VGG-16) Accuracy')
plt.legend()
plt.show()


In [ ]:
print("Model 4: Train Performance (with Augmentation)")
# Use the generator for performance check to stay consistent
display(model_performance_classification(model_4, X_train_vgg, y_train))

plot_confusion_matrix(model_4, X_train_vgg, y_train)

print("Model 4: Validation Performance")
display(model_performance_classification(model_4, X_val_vgg, y_val))

plot_confusion_matrix(model_4, X_val_vgg, y_val)


In [ ]:
# Visualize and predict for some random index to cross verify
# Note: We use X_val for display (0-1 range) but X_val_vgg for the model prediction
index = 60
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

# Make prediction using the correctly preprocessed VGG array
prediction = model_4.predict(X_val_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])

# Visualize and predict for another random index
index = 0
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

prediction = model_4.predict(X_val_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])

# Visualize and predict for another random index
index = 10
plt.figure(figsize=(2,2))
plt.imshow(X_val[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

prediction = model_4.predict(X_val_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_val.iloc[index].values[0])


<p>We can see that model 4 with VGG and data augmentation produces accurate results for the samples I had chosen earlier</p>


<h1><strong>Model Performance Comparison and Final Model Selection</strong></h1>


In [ ]:
models = {
    "Simple CNN": (cnn_model_1, X_val, y_val),
    "VGG-16 Base": (model_2, X_val_vgg, y_val),
    "VGG-16 + Custom FFNN": (model_3, X_val_vgg, y_val),
    "VGG-16 + FFNN + Augmentation": (model_4, X_val_vgg, y_val)
}

comparison_list = []

for name, (model, predictors, target) in models.items():
    perf = model_performance_classification(model, predictors, target)
    perf.index = [name]
    comparison_list.append(perf)

comparison_df = pd.concat(comparison_list)
display(comparison_df.sort_values(by="F1 Score", ascending=False))


<p>Based on the above performances I am going ot choose model 4 because it has data augemented and provided accurate results</p>


<h2>Test Performance</h2>


In [ ]:
print("Evaluating Final Model (Model 4) on Test Set...")
test_performance = model_performance_classification(model_4, X_test_vgg, y_test)
display(test_performance)

plot_confusion_matrix(model_4, X_test_vgg, y_test)


In [ ]:
# Visualize and predict for some random index to cross verify
# Note: We use X_val for display (0-1 range) but X_val_vgg for the model prediction
index = 60
plt.figure(figsize=(2,2))
plt.imshow(X_test[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

# Make prediction using the correctly preprocessed VGG array
prediction = model_4.predict(X_test_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_test.iloc[index].values[0])

# Visualize and predict for another random index
index = 0
plt.figure(figsize=(2,2))
plt.imshow(X_test[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

prediction = model_4.predict(X_test_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_test.iloc[index].values[0])

# Visualize and predict for another random index
index = 10
plt.figure(figsize=(2,2))
plt.imshow(X_test[index])
plt.title(f"Image at index {index}")
plt.axis('off')
plt.show()

prediction = model_4.predict(X_test_vgg[index].reshape(1, 224, 224, 3))
predicted_label = 1 if prediction[0][0] > 0.5 else 0
print('Predicted Label:', predicted_label)
print('True Label:', y_test.iloc[index].values[0])


<p>We can see that with cross verification against sample images in test set, our model 4 predicts very well.</p>


<h1><strong>Actionable Insights &amp; Recommendations</strong></h1>


<p>Actionable Insights:</p>
<ul>
<li><p>Model 4 is the Production Candidate: While simpler CNNs achieved high training accuracy, they suffered from "memorization" (overfitting). Model 4, which combines VGG-16 transfer learning with Data Augmentation, is the most robust. It has learned to identify the features of a helmet rather than just the specific pixels of the training set.</p>
</li>
<li><p>Data Augmentation: With a relatively small dataset (630 images), the model initially struggled with variation. Augmentation (flips, rotations, shifts) essentially "taught" the model that a helmet is still a helmet even if the worker is looking away or the camera is at an angle.</p>
</li>
<li><p>The "Safety Gap" (Recall vs. Accuracy): Simple accuracy score can be misleading. In safety, we care most about Recall—the ability to catch every single instance of a missing helmet. A "False Alarm" (Precision error) costs a few seconds of a supervisor's time, but a "Missed Violation" (Recall error) can lead to a fatal accident.</p>
</li>
</ul>
<p>Recommendations</p>
<ol>
<li><p>Implement a "Safety-First" Threshold: Instead of the standard 0.5 probability threshold, lower the threshold to 0.3 or 0.2. This will make the model more "sensitive" (higher Recall), ensuring that any ambiguous case is flagged for human review.</p>
</li>
<li><p>Expand Dataset Diversity: The model currently performs best in conditions similar to the training data. To improve reliability, SafeGuard should collect and label images from"
a. Different Lighting: Night shifts, high-glare noon sun, and rainy conditions.
b. Different PPE Colors: Ensure the model recognizes blue, white, yellow, and orange helmets equally well.</p>
</li>
<li><p>Complex Backgrounds: Images with heavy machinery or scaffolding in the background to reduce false positives.
Full PPE Suite: The architecture used for HelmNet (VGG-16 + FFNN) can be easily adapted to detect other safety gear. Use the same pipeline to train detectors for:
a. High-visibility safety vests.
b. Protective eyewear/goggles.
c. Safety harnesses for workers at heights.</p>
</li>
</ol>


<p><font>Power Ahead!</font></p>
<hr></hr>
